In [0]:
!pwd

In [0]:
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
base_volume = f"/Volumes/{catalog_name}/default/fundamentals_demo"

files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

for f in files:
    dbutils.fs.cp(f"file:/Workspace/Users/darsh2115@gmail.com/demo/fundamentals/{f}", f"{base_volume}/{f}")

In [0]:
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
base_volume = f"/Volumes/{catalog_name}/default/fundamentals_demo"

df = spark.read.load(f"/Volumes/{catalog_name}/default/fundamentals_demo/*_edited.csv", format="csv")

# Defining the Schema

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", StringType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType()),
])

df = spark.read.load(f"/Volumes/demo/default/fundamentals_demo/*_edited.csv", format="csv", schema=orderSchema)

df.display()

In [0]:
import pyspark.sql.functions as F

In [0]:
df = df.dropDuplicates()
df = df.withColumn("Tax", F.col("UnitPrice") * 0.08)
df = df.withColumn("Tax", F.col('Tax').cast('float'))
df.display()

In [0]:
df.printSchema()

# Define Customer DF

In [0]:
customers_df = df.select("CustomerName", "Email", "Item", "Quantity")
customers_df = customers_df.withColumn("FirstName", F.split(customers_df["CustomerName"], " ").getItem(0))
customers_df = customers_df.withColumn("LastName", F.split(customers_df["CustomerName"], " ").getItem(1))
customers_df.printSchema()

In [0]:
customers_df.display()

In [0]:
customers_df.count(), customers_df.distinct().count()

# Createing Product Sales Dataframe

In [0]:
productSales = df.select("Item", "Quantity").groupBy("Item").sum()

productSales.display()

# Aggregating Yearly Sales

In [0]:
yearlySales = df.select(F.year("OrderDate").alias("OrderYear")).groupBy('OrderYear').count().orderBy("OrderYear")

yearlySales.display()